In [ ]:
import random

class VacuumEnv:
    def __init__(self, rooms=("A", "B"), dirty=("A", "B"), start="A",
                 p_dirty_again=0.0, p_suck_fail=0.0, seed=None):
        self.rooms = list(rooms)
        self.status = {r: ("Dirty" if r in dirty else "Clean") for r in self.rooms}
        self.loc = start
        self.p_dirty_again = p_dirty_again   # احتمال إن غرفة نضيفة تتوسخ تاني
        self.p_suck_fail = p_suck_fail       # احتمال إن Suck يفشل
        self.rng = random.Random(seed)

    def percept(self):
        return (self.loc, self.status[self.loc])

    def step(self, action):
        i = self.rooms.index(self.loc)
        move_cost = 0.0
        if action == "Suck":
            if self.rng.random() >= self.p_suck_fail:
                self.status[self.loc] = "Clean"
        elif action == "Right" and i < len(self.rooms) - 1:
            self.loc = self.rooms[i + 1]; move_cost = 0.5
        elif action == "Left" and i > 0:
            self.loc = self.rooms[i - 1]; move_cost = 0.5
        # NoOp: مفيش حاجة تحصل
        for r in self.rooms:
            if self.status[r] == "Clean" and self.rng.random() < self.p_dirty_again:
                self.status[r] = "Dirty"
        clean_rooms = sum(s == "Clean" for s in self.status.values())
        return clean_rooms - move_cost



def run(agent, env, steps=20, verbose=False):
    total = 0
    for t in range(steps):
        p = env.percept()
        a = agent(p)
        r = env.step(a)
        total += r
        if verbose:
            print(f"t={t:2d} percept={p!s:18} action={a:6} reward={r:4} status={env.status}")
    return total